In [260]:
%load_ext autoreload
%autoreload 2

import sys
import os

current_dir = os.getcwd()

project_root = os.path.abspath(os.path.join(current_dir, ".."))

if project_root not in sys.path:
    sys.path.append(project_root)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [261]:
from src import *
from utils import *
from scipy.spatial import cKDTree

In [262]:
N=6400
l=4
K=50
l_grad=3
K_grad=30
kappa = 3
delta = 1e-5

In [263]:
np.random.seed(0)

In [264]:
theta, phi = sp.symbols('theta phi', real=True)

R = 0.5

x_sym = R * sp.sin(phi) * sp.cos(theta)
y_sym = R * sp.sin(phi) * sp.sin(theta)
z_sym = R * sp.cos(phi)

manifold = Manifold([theta, phi], [x_sym, y_sym, z_sym])
manifold.compute()

theta_range = (0, 2*np.pi)
phi_range = (0, np.pi)

phi_if = np.pi/2

num_interface = int(np.round(np.sqrt(np.pi * N)))
num_interior = N - num_interface

# sample interior
manifold.sample([theta_range, phi_range], num_interior)

# sample interface
x_sym_if = sp.sin(phi_if) * sp.cos(theta)
y_sym_if = sp.sin(phi_if) * sp.sin(theta)
z_sym_if = 0

interface = Manifold([theta], [x_sym_if, y_sym_if, z_sym_if])
interface.sample([theta_range], num_interface)

manifold.params = np.vstack([
    manifold.params,
    np.insert(interface.params, 1, values=phi_if, axis=1),
])

manifold.points = np.vstack([
    manifold.points,
    interface.points,
])

id_interior = np.arange(num_interior)
id_if = np.arange(num_interior, N)

# interface direction
n_vecs = np.zeros((num_interface, manifold.n)) # shape: (num_boundary, n)

for i in range(num_interface):
    n_vecs[i, :] = [0.0, 0.0, 1.0] # pointing the upper manifold

z_coords = manifold.points[id_interior, 2]
id_upper = id_interior[z_coords > 0]
id_lower = id_interior[z_coords < 0]

In [265]:
alpha = 1

beta_upper = 10
beta_lower = 1

In [266]:
# Manufacture solution

u_sym_upper = x_sym * y_sym * sp.sin(4 * sp.pi * z_sym)
u_sym_lower = x_sym * y_sym * sp.cos(4 * sp.pi * z_sym)

u_lap_sym_upper = manifold.get_laplacian(u_sym_upper)
u_lap_sym_lower = manifold.get_laplacian(u_sym_lower)

u_grad_sym_upper = manifold.get_gradient(u_sym_upper)
u_grad_sym_lower = manifold.get_gradient(u_sym_lower)

# f = -beta * \Delta u + alpha * u 
f_sym_upper = -beta_upper * u_lap_sym_upper + alpha * u_sym_upper
f_sym_lower = -beta_lower * u_lap_sym_lower + alpha * u_sym_lower

tt = manifold.params[:, 0]
pp = manifold.params[:, 1]

# Lambdify 
f_func_upper = sp.lambdify((theta, phi), f_sym_upper, 'numpy')
f_func_lower = sp.lambdify((theta, phi), f_sym_lower, 'numpy')

u_func_upper = sp.lambdify((theta, phi), u_sym_upper, 'numpy')
u_func_lower = sp.lambdify((theta, phi), u_sym_lower, 'numpy')

u_grad_func_upper = sp.lambdify((theta, phi), u_grad_sym_upper, 'numpy')
u_grad_func_lower = sp.lambdify((theta, phi), u_grad_sym_lower, 'numpy')

# interior points
u_vals = np.zeros(N)
f_vals = np.zeros(N)

u_vals[id_upper] = u_func_upper(tt[id_upper], pp[id_upper])
u_vals[id_lower] = u_func_lower(tt[id_lower], pp[id_lower])

f_vals[id_upper] = f_func_upper(tt[id_upper], pp[id_upper])
f_vals[id_lower] = f_func_lower(tt[id_lower], pp[id_lower])

u_lap_vals_upper = (alpha * u_vals[id_upper] - f_vals[id_upper]) / beta_upper
u_lap_vals_lower = (alpha * u_vals[id_lower] - f_vals[id_lower]) / beta_lower

#-- INTERFACE JUMP CONDITIONS --#

tt_if = tt[id_if]
pp_if = pp[id_if]

# 1. q = u+ - u-
u_vals_if_upper = u_func_upper(tt_if, pp_if)
u_vals_if_lower = u_func_lower(tt_if, pp_if)
q_vals = u_vals_if_upper - u_vals_if_lower # shape: (num_interface,)

# 2. psi = beta^+ * \nabla u^+ * n - beta^- * \nabla u^- * n [cite: 52]
u_grad_vals_if_upper = np.array(u_grad_func_upper(tt_if, pp_if)).squeeze().T # shape: (num_interface, 3)
u_grad_vals_if_lower = np.array(u_grad_func_lower(tt_if, pp_if)).squeeze().T 

flux_upper = beta_upper * np.sum(n_vecs * u_grad_vals_if_upper, axis=1)
flux_lower = beta_lower * np.sum(n_vecs * u_grad_vals_if_lower, axis=1)
psi_vals = flux_upper - flux_lower # shape: (num_interface,)

In [267]:
num_upper = len(id_upper)
num_lower = len(id_lower)

id_upper_if = np.concatenate([id_upper, id_if])
tree_upper = cKDTree(manifold.points[id_upper_if])

L_upper = np.zeros((num_upper, num_upper + num_interface))
D_n_upper = np.zeros((num_interface, num_upper + num_interface))

for i, u_id in enumerate(id_upper):
    _, stencil_ids = tree_upper.query(manifold.points[u_id], K)

    weights_lap = get_operator_weights(
        stencil=tree_upper.data[stencil_ids],
        tangent_basis=manifold.get_local_basis(manifold.params[u_id])[0],
        operator='lap',
        kappa=kappa,
        l=l,
        delta=delta,
    ) # shape: (1, K)

    L_upper[i, stencil_ids] = weights_lap[0, :]

for i, i_id in enumerate(id_if):
    _, stencil_ids = tree_upper.query(manifold.points[i_id], K_grad)

    weights_grad = get_operator_weights(
        stencil=tree_upper.data[stencil_ids],
        tangent_basis=manifold.get_local_basis(manifold.params[i_id])[0],
        operator='grad',
        kappa=kappa,
        l=l_grad,
        delta=delta,
    ) # shape: (n, K)

    n_vec = n_vecs[i]
    weights_grad_n = n_vec @ weights_grad # shape: (K,)

    D_n_upper[i, stencil_ids] = weights_grad_n

id_lower_if = np.concatenate([id_lower, id_if])
tree_lower = cKDTree(manifold.points[id_lower_if])

L_lower = np.zeros((num_lower, num_lower + num_interface))
D_n_lower = np.zeros((num_interface, num_lower + num_interface))

for i, l_id in enumerate(id_lower):
    _, stencil_ids = tree_lower.query(manifold.points[l_id], K)

    weights_lap = get_operator_weights(
        stencil=tree_lower.data[stencil_ids],
        tangent_basis=manifold.get_local_basis(manifold.params[l_id])[0],
        operator='lap',
        kappa=kappa,
        l=l,
        delta=delta,
    ) # shape: (1, K)

    L_lower[i, stencil_ids] = weights_lap[0, :]

for i, i_id in enumerate(id_if):
    _, stencil_ids = tree_lower.query(manifold.points[i_id], K_grad)

    weights_grad = get_operator_weights(
        stencil=tree_lower.data[stencil_ids],
        tangent_basis=manifold.get_local_basis(manifold.params[i_id])[0],
        operator='grad',
        kappa=kappa,
        l=l_grad,
        delta=delta,
    ) # shape: (n, K)

    n_vec = n_vecs[i]
    weights_grad_n = n_vec @ weights_grad # shape: (K,)

    D_n_lower[i, stencil_ids] = weights_grad_n

In [268]:
# ==========================================
# 算子离散正确性验证 (Validation)
# ==========================================

# 1. 组装输入精确解向量 (内部点拼接界面点)
u_exact_upper_if = np.concatenate([u_vals[id_upper], u_vals_if_upper])
u_exact_lower_if = np.concatenate([u_vals[id_lower], u_vals_if_lower])

# 2. 验证 L_upper 和 L_lower
lap_pred_upper = L_upper @ u_exact_upper_if
err_lap_upper = np.max(np.abs(lap_pred_upper - u_lap_vals_upper))

lap_pred_lower = L_lower @ u_exact_lower_if
err_lap_lower = np.max(np.abs(lap_pred_lower - u_lap_vals_lower))

# 3. 验证 D_n_upper (\nabla u^+ \cdot n)
# n_vecs 是你在外部循环定义的 [0, 0, 1] 法向量
exact_dn_upper = np.sum(n_vecs * u_grad_vals_if_upper, axis=1)
dn_pred_upper = D_n_upper @ u_exact_upper_if
err_dn_upper = np.max(np.abs(dn_pred_upper - exact_dn_upper))

# 4. 验证 D_n_lower (\nabla u^- \cdot n)
exact_dn_lower = np.sum(n_vecs * u_grad_vals_if_lower, axis=1)
dn_pred_lower = D_n_lower @ u_exact_lower_if
err_dn_lower = np.max(np.abs(dn_pred_lower - exact_dn_lower))

print(f"Max Error for L_upper:   {err_lap_upper:.4e}")
print(f"Max Error for L_lower:   {err_lap_lower:.4e}")
print(f"Max Error for D_n_upper: {err_dn_upper:.4e}")
print(f"Max Error for D_n_lower: {err_dn_lower:.4e}")

Max Error for L_upper:   1.0032e+00
Max Error for L_lower:   1.5291e+00
Max Error for D_n_upper: 1.5671e+00
Max Error for D_n_lower: 6.4151e+00


In [269]:
# =========================================================
# 全局分块矩阵组装与求解 (Block Matrix Assembly & Solve)
# =========================================================

N_total = num_upper + num_interface + num_lower + num_interface

G = np.zeros((N_total, N_total))
rhs = np.zeros(N_total)

# 定义变量在全局系统中的列索引范围
# 未知量向量排列为: U = [u_up, u_if_up, u_low, u_if_low]^T
col_up     = slice(0, num_upper)
col_if_up  = slice(num_upper, num_upper + num_interface)
col_low    = slice(num_upper + num_interface, num_upper + num_interface + num_lower)
col_if_low = slice(num_upper + num_interface + num_lower, N_total)

# 定义方程在全局系统中的行索引范围
row_up     = col_up      # 上半球内部控制方程
row_if_dir = col_if_up   # 界面解跳跃条件 (Dirichlet)
row_low    = col_low     # 下半球内部控制方程
row_if_neu = col_if_low  # 界面通量跳跃条件 (Neumann)

# 1. 组装上半球内部点方程: -\beta^+ \Delta u^+ + \alpha u^+ = f^+
G[row_up, col_up]    = -beta_upper * L_upper[:, :num_upper] + alpha * np.eye(num_upper)
G[row_up, col_if_up] = -beta_upper * L_upper[:, num_upper:]
rhs[row_up]          = f_vals[id_upper]

# 2. 组装下半球内部点方程: -\beta^- \Delta u^- + \alpha u^- = f^-
G[row_low, col_low]    = -beta_lower * L_lower[:, :num_lower] + alpha * np.eye(num_lower)
G[row_low, col_if_low] = -beta_lower * L_lower[:, num_lower:]
rhs[row_low]           = f_vals[id_lower]

# 3. 组装界面条件 1: 解的跳跃 u^+ - u^- = q
G[row_if_dir, col_if_up]  = np.eye(num_interface)
G[row_if_dir, col_if_low] = -np.eye(num_interface)
rhs[row_if_dir]           = q_vals

# 4. 组装界面条件 2: 通量的跳跃 \beta^+ D_n^+ u^+ - \beta^- D_n^- u^- = \psi
G[row_if_neu, col_up]     = beta_upper * D_n_upper[:, :num_upper]
G[row_if_neu, col_if_up]  = beta_upper * D_n_upper[:, num_upper:]
G[row_if_neu, col_low]    = -beta_lower * D_n_lower[:, :num_lower]
G[row_if_neu, col_if_low] = -beta_lower * D_n_lower[:, num_lower:]
rhs[row_if_neu]           = psi_vals

# =========================================================
# 求解与误差分析
# =========================================================

print("Solving the linear system...")
U_sol = np.linalg.solve(G, rhs)

# 提取各部分的解
u_pred_upper    = U_sol[col_up]
u_pred_if_upper = U_sol[col_if_up]
u_pred_lower    = U_sol[col_low]
u_pred_if_lower = U_sol[col_if_low]

# 将内部点和界面点拼接，与精确解做对比
u_pred_upper_all = np.concatenate([u_pred_upper, u_pred_if_upper])
u_exact_upper_all = np.concatenate([u_vals[id_upper], u_vals_if_upper])
error_upper = np.max(np.abs(u_pred_upper_all - u_exact_upper_all))

u_pred_lower_all = np.concatenate([u_pred_lower, u_pred_if_lower])
u_exact_lower_all = np.concatenate([u_vals[id_lower], u_vals_if_lower])
error_lower = np.max(np.abs(u_pred_lower_all - u_exact_lower_all))

print(f"L-infinity Error (Upper domain + Interface): {error_upper:.4e}")
print(f"L-infinity Error (Lower domain + Interface): {error_lower:.4e}")

Solving the linear system...


LinAlgError: Singular matrix

In [ ]:

# # ---------------------------------------------------------
# # 1. 建立全局自由度映射 (Degrees of Freedom Mapping)
# # ---------------------------------------------------------
# N_up = len(id_upper)
# N_low = len(id_lower)
# N_if = len(id_if)
# N_total = N_up + N_low + 2 * N_if

# # 建立物理点索引到矩阵列索引的映射
# col_map_upper = {}
# col_map_lower = {}

# for i, p in enumerate(id_upper): col_map_upper[p] = i
# for i, p in enumerate(id_lower): col_map_lower[p] = N_up + i

# # 界面点分裂为上侧极限和下侧极限两个未知量
# for i, p in enumerate(id_if):
#     col_map_upper[p] = N_up + N_low + i          # u^+ 的全局列索引
#     col_map_lower[p] = N_up + N_low + N_if + i   # u^- 的全局列索引

# # ---------------------------------------------------------
# # 2. 构建单侧 KDTree
# # ---------------------------------------------------------
# pts_upper_if = np.concatenate([id_upper, id_if])
# pts_lower_if = np.concatenate([id_lower, id_if])

# tree_upper = cKDTree(manifold.points[pts_upper_if])
# tree_lower = cKDTree(manifold.points[pts_lower_if])

# # ---------------------------------------------------------
# # 3. 组装全局稠密矩阵 G 和右端项 rhs
# # ---------------------------------------------------------
# G = np.zeros((N_total, N_total))
# rhs = np.zeros(N_total)
# row_idx = 0

# # --- 3.1 上半球内部点 (Laplacian) ---
# for p in id_upper:
#     p_coords = manifold.points[p]
#     tangent_basis = manifold.get_local_basis(manifold.params[p])[0]
    
#     _, nbr_idx_in_tree = tree_upper.query(p_coords, k=K)
#     actual_nbrs = pts_upper_if[nbr_idx_in_tree]
#     stencil = manifold.points[actual_nbrs]
    
#     w_lap = get_operator_weights(stencil, tangent_basis, kappa=kappa, l=l, delta=delta, operator='lap')
    
#     for j, nbr in enumerate(actual_nbrs):
#         G[row_idx, col_map_upper[nbr]] += -beta_upper * w_lap[0, j]
#         if nbr == p:
#             G[row_idx, col_map_upper[nbr]] += alpha
            
#     rhs[row_idx] = f_vals[p]
#     row_idx += 1

# # --- 3.2 下半球内部点 (Laplacian) ---
# for p in id_lower:
#     p_coords = manifold.points[p]
#     tangent_basis = manifold.get_local_basis(manifold.params[p])[0]
    
#     _, nbr_idx_in_tree = tree_lower.query(p_coords, k=K)
#     actual_nbrs = pts_lower_if[nbr_idx_in_tree]
#     stencil = manifold.points[actual_nbrs]
    
#     w_lap = get_operator_weights(stencil, tangent_basis, kappa=kappa, l=l, delta=delta, operator='lap')
    
#     for j, nbr in enumerate(actual_nbrs):
#         G[row_idx, col_map_lower[nbr]] += -beta_lower * w_lap[0, j]
#         if nbr == p:
#             G[row_idx, col_map_lower[nbr]] += alpha
            
#     rhs[row_idx] = f_vals[p]
#     row_idx += 1

# # --- 3.3 界面条件 1: 解的跳跃 (Dirichlet) ---
# # 方程: u^+ - u^- = q
# for i, p in enumerate(id_if):
#     G[row_idx, col_map_upper[p]] = 1.0
#     G[row_idx, col_map_lower[p]] = -1.0
#     rhs[row_idx] = q_vals[i]
#     row_idx += 1

# # --- 3.4 界面条件 2: 通量的跳跃 (Neumann) ---
# # 方程: beta^+ \nabla u^+ \cdot n - beta^- \nabla u^- \cdot n = psi [cite: 52]
# for i, p in enumerate(id_if):
#     p_coords = manifold.points[p]
#     tangent_basis = manifold.get_local_basis(manifold.params[p])[0]
#     n_vec = n_vecs[i]
    
#     # 提取 Upper 侧的方向导数权重
#     _, nbr_idx_up = tree_upper.query(p_coords, k=K_grad)
#     actual_nbrs_up = pts_upper_if[nbr_idx_up]
#     stencil_up = manifold.points[actual_nbrs_up]
    
#     w_grad_up = get_operator_weights(stencil_up, tangent_basis, kappa=kappa, l=l_grad, delta=delta, operator='grad')
#     w_dir_up = np.dot(n_vec, w_grad_up) # 内积得到方向导数权重
    
#     for j, nbr in enumerate(actual_nbrs_up):
#         G[row_idx, col_map_upper[nbr]] += beta_upper * w_dir_up[j]
        
#     # 提取 Lower 侧的方向导数权重
#     _, nbr_idx_low = tree_lower.query(p_coords, k=K_grad)
#     actual_nbrs_low = pts_lower_if[nbr_idx_low]
#     stencil_low = manifold.points[actual_nbrs_low]
    
#     w_grad_low = get_operator_weights(stencil_low, tangent_basis, kappa=kappa, l=l_grad, delta=delta, operator='grad')
#     w_dir_low = np.dot(n_vec, w_grad_low)
    
#     for j, nbr in enumerate(actual_nbrs_low):
#         G[row_idx, col_map_lower[nbr]] += -beta_lower * w_dir_low[j]
        
#     rhs[row_idx] = psi_vals[i]
#     row_idx += 1

# # ---------------------------------------------------------
# # 4. 求解稠密线性系统
# # ---------------------------------------------------------
# print("Solving the dense linear system...")
# U_sol = np.linalg.solve(G, rhs)

# # ---------------------------------------------------------
# # 5. 提取预测解并重构回物理空间顺序
# # ---------------------------------------------------------
# u_pred_upper = U_sol[[col_map_upper[p] for p in id_upper]]
# u_pred_lower = U_sol[[col_map_lower[p] for p in id_lower]]
# u_pred_if_upper = U_sol[[col_map_upper[p] for p in id_if]]
# u_pred_if_lower = U_sol[[col_map_lower[p] for p in id_if]]

# # 计算误差等评估指标
# error_upper = np.max(np.abs(u_pred_upper - u_vals[id_upper]))
# error_lower = np.max(np.abs(u_pred_lower - u_vals[id_lower]))
# print(f"L-infinity Error (Upper): {error_upper:.4e}")
# print(f"L-infinity Error (Lower): {error_lower:.4e}")